# Anchor extraction demo

Core engine: the LLM returns **verbatim boundary anchors only**; Python resolves offsets, slices text, groups segments by role, and stitches across chunks.

An **extraction profile** (detection prompt + optional unit-boundary regex) decides which logical units to extract. Bundled profiles: `anchor_extract/prompts/profiles/`.

The LLM provider (Azure OpenAI or Anthropic) is chosen from `.env` (`ANCHOR_LLM_PROVIDER=auto`).

Set `PDF_PATH` to a text-extractable PDF. Docs: `docs/core_pipeline.md`, `docs/profiles.md`.


In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from anchor_extract import extract_pdf, build_anchor_system_prompt
    
from anchor_extract.pipeline import extract_document, to_requirements_json, save_json
from anchor_extract.settings import EXAMPLE_AI_RMF_BOUNDARY_PATTERN

load_dotenv()

# HIPAA (matches PDF_PATH)
PDF_PATH = Path("pdf/hipaa-simplification-201303.pdf")
DETECTION_PROMPT_PATH = Path("anchor_extract/prompts/profiles/hipaa.txt")
BOUNDARY_PATTERN = None

# Alternative: NIST AI RMF Playbook
# PDF_PATH = Path("pdf/AI_RMF_Playbook.pdf")
# DETECTION_PROMPT_PATH = Path("anchor_extract/prompts/profiles/ai_rmf_playbook.txt")
# BOUNDARY_PATTERN = EXAMPLE_AI_RMF_BOUNDARY_PATTERN


In [5]:
START_PAGE = 11
END_PAGE = 17

doc = extract_pdf(str(PDF_PATH), start_page=START_PAGE, end_page=END_PAGE)

blocks_df = pd.DataFrame([{
    "page": b.page,
    "char_start": b.char_start,
    "char_end": b.char_end,
    "text_preview": b.text[:120],
} for b in doc.blocks[:15]])
blocks_df

,page,char_start,char_end,text_preview
0,11,0,24,§ 160.102 Applicability.
1,11,25,184,"(a) Except as otherwise provided, the standard..."
2,11,185,203,(1) A health plan.
3,11,204,236,(2) A health care clearinghouse.
4,11,237,380,(3) A health care provider who transmits any h...
5,11,381,524,"(b) Where provided, the standards, requirement..."
6,11,525,809,(c) To the extent required under the Social Se...
7,11,810,907,"[65 FR 82798, Dec. 28, 2000, as amended at 67 ..."
8,11,908,930,§ 160.103 Definitions.
9,11,931,1012,"Except as otherwise provided, the following de..."


In [3]:
detection_prompt = DETECTION_PROMPT_PATH.read_text(encoding="utf-8")
system_prompt = build_anchor_system_prompt(detection_prompt)

if system_prompt:
    print("Prompth loaded correctly")
else:
    print("Error loading prompt")

Prompth loaded correctly


In [6]:
extraction = extract_document(
    str(PDF_PATH),
    detection_prompt,
    doc=doc,
    start_page=START_PAGE,
    end_page=END_PAGE,
    requirement_boundary_pattern=BOUNDARY_PATTERN,
    verbose=True,)

Chunk budget: 12000 input tok (override; ctx 200000, prompt~3144, out 8000)
[iter   1] blocks    0-212  | pages  11-17  | est  6676 tok | target 12000 | shrinks 0 ... 4 reqs (4 ok, 0 trunc) | stop=tool_use | 15.813s
  -> stitched 4 requirement(s) | no pending

Done. 1 batch(es) in 15.8s | 1/1 ok API call(s) | 4 requirement(s) stitched.


In [7]:
results_df = pd.DataFrame([{
    "requirement_id": r.requirement_id,
    "status": r.status,
    "verbatim_match": r.verbatim_match,
    "end_resolved": r.end_resolved,
    "n_segments": r.n_segments,
    "n_segments_resolved": r.n_segments_resolved,
    "doc_offset_start": r.doc_offset_start,
    "doc_offset_end": r.doc_offset_end,
} for r in extraction.requirements])
results_df

,requirement_id,status,verbatim_match,end_resolved,n_segments,n_segments_resolved,doc_offset_start,doc_offset_end
0,160.102,complete,True,True,1,1,0,907
1,160.104,complete,True,True,1,1,24490,25731
2,160.105,complete,True,True,1,1,25732,26428
3,160.201,complete,True,True,1,1,26463,26698


In [ ]:
for r in extraction.requirements:
    if r.status != "complete" or not r.verbatim_match:
        continue
    print("===", r.requirement_id, "===")
    print("requirement:", r.requirement_text.strip()[:200], "...")
    print("context:", (r.context_text or "").strip()[:200], "...")
    print("questionnaire:", (r.questionnaire_text or "").strip()[:200], "...")

In [ ]:
for r in extraction.requirements:
    if r.n_segments != 1 or r.doc_offset_start < 0:
        print(r.requirement_id, "invariant: n/a (multi-span or unresolved)")
        continue
    ok = doc.full_text[r.doc_offset_start:r.doc_offset_end] == r.original_text
    print(r.requirement_id, "invariant:", "ok" if ok else "BAD")

In [ ]:
out = Path("outputs/requirements.json")
save_json(to_requirements_json("HIPAA", doc, extraction), out)
print("Wrote", out.resolve())
